# 🚗 Driver Behavior Analysis using Smartphone Sensors & Computer Vision

**Author:** Ajay Bhagwan Gaikwad  
**Dataset:** DriverSVT (Zenodo — 17.5M rows, 633 drivers, 9 behavior classes)  
**GitHub:** [AjayGaikwad-07/driver-behavior-ml](https://github.com/AjayGaikwad-07/driver-behavior-ml)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AjayGaikwad-07/driver-behavior-ml/blob/main/driver_behavior_colab_FINAL.ipynb)

---

## 📋 Project Workflow

```
STAGE 1        STAGE 2       STAGE 3        STAGE 4        STAGE 5
Problem    ->  Environment   Dataset    ->  Data Quality   EDA
Statement      Setup         Loading        Check          Analysis

STAGE 6        STAGE 7       STAGE 8        STAGE 9        STAGE 10
Feature    ->  Group-Safe    Multi-Model    Confusion      Per-Class
Engineering    Train/Test    Benchmark      Matrix         Metrics

STAGE 11       STAGE 12      STAGE 13       STAGE 14
Feature    ->  Feature       Trip Risk      Results
Importance     Ablation      Scoring        Summary
```

**Primary Metric:** Macro-F1 (treats all 9 classes equally -- critical for imbalanced safety data)  
**Secondary Metrics:** Balanced Accuracy, Per-class Precision/Recall

---
## Stage 1 -- Problem Statement & Motivation

Road accidents due to distracted, drowsy, or reckless drivers cause thousands of deaths annually. Traditional driver-monitoring systems require expensive in-car hardware (IR cameras, steering torque sensors).

**Research Question:** Can we detect dangerous driving behaviors using only a smartphone's built-in sensors (accelerometer, gyroscope, GPS) + lightweight computer vision -- no expensive hardware needed?

### 9-Class Classification Problem
| Class | Description | Risk Level |
|---|---|---|
| Normal | Safe, attentive driving | Low |
| Distraction | General inattention | Medium |
| Drowsiness | Fatigue-related impairment | High |
| Using phone | Phone held in hand | High |
| No attention | Looking away from road | High |
| Smoking | Cigarette while driving | Medium |
| Eating | Food/drink while driving | Medium |
| Unfastened belt | Seatbelt not worn | Medium |
| Camera off | Driver monitoring disabled | High |

### Why NOT use raw accuracy?
With ~27% 'Normal' class dominance, a model that always predicts 'Normal' gets high accuracy but catches **zero** dangerous events -- useless for safety. **Macro-F1** is used instead.

---
## Stage 2 -- Environment Setup

In [ ]:
# Hardware Check
import sys, os
print(f"Python: {sys.version.split()[0]}")
try:
    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    else:
        print("No GPU -- running on CPU (fine for tabular ML)")
except ImportError:
    print("PyTorch not yet installed")

In [ ]:
!pip install -q --upgrade pip
!pip install -q ultralytics mediapipe opencv-python-headless scikit-learn pandas numpy matplotlib seaborn joblib
print("All packages installed!")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import urllib.request
import time

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix,
    precision_recall_fscore_support
)

# Plot style
STYLE = "seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default"
plt.style.use(STYLE)
plt.rcParams.update({
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.dpi': 120,
})
PALETTE = ['#4361EE','#F72585','#7209B7','#3A0CA3','#4CC9F0',
           '#06D6A0','#FFB703','#FB8500','#EF476F']

# Constants
RAW_FEATURES = [
    'speed', 'gforce', 'acc_X', 'acc_Y', 'acc_Z',
    'accelerometer_data_X', 'accelerometer_data_Y', 'accelerometer_data_Z',
    'gyroscope_data_X', 'gyroscope_data_Y', 'gyroscope_data_Z',
    'perclos', 'mouth_open_ratio'
]
TARGET = 'dangerousstate'
GROUP  = 'userid'

print("Imports done, style configured.")

---
## Stage 3 -- Dataset Loading

**DriverSVT** -- collected from **633 drivers** via smartphones mounted in their vehicles.  
Working sample: **~580K rows** across **491 drivers** (~68 MB balanced class subsample).

| Column | Description | Unit |
|---|---|---|
| speed | Vehicle speed | km/h |
| gforce | Lateral g-force | G |
| acc_X/Y/Z | Linear acceleration | m/s2 |
| accelerometer_data_X/Y/Z | Total acceleration (gravity included) | m/s2 |
| gyroscope_data_X/Y/Z | Angular velocity | deg/s |
| perclos | Eye closure % (fatigue proxy) | 0-1 |
| mouth_open_ratio | Mouth openness (yawn proxy) | 0-1 |
| userid | Driver ID | int |
| dangerousstate | Target label (9 classes) | text |

In [ ]:
# Clone repo and download model
REPO_URL = "https://github.com/AjayGaikwad-07/driver-behavior-ml.git"
REPO_DIR = Path("driver-behavior-ml")

if not REPO_DIR.exists() and not Path("day3_prepare_dataset.py").exists():
    print("Cloning repo ...")
    !git clone $REPO_URL
    os.chdir(REPO_DIR)
    print("Working dir:", os.getcwd())
else:
    print("Repo already present.")

for folder in ["data/demo", "data/raw", "models", "reports"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

TASK_PATH = Path("models/face_landmarker.task")
if not TASK_PATH.exists():
    print("Downloading face_landmarker.task ...")
    url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
    urllib.request.urlretrieve(url, TASK_PATH)
    print(f"Done! ({TASK_PATH.stat().st_size / 1e6:.2f} MB)")
else:
    print(f"face_landmarker.task ready ({TASK_PATH.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# Dataset Discovery and Loading
CANDIDATE_PATHS = [
    Path("day3_balanced_group_split_fast/sampled_driver_data_balanced_group_split.csv"),
    Path("day3_working/sampled_driver_data.csv"),
    Path("data/raw/DriverSVT/DriverSVT.csv"),
    Path("/content/drive/MyDrive/driver-behavior-ai/sampled_driver_data_balanced_group_split.csv"),
    Path("/content/drive/MyDrive/DriverSVT/DriverSVT.csv"),
    Path("sampled_driver_data_balanced_group_split.csv")
]

DATASET_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), None)

if DATASET_PATH is None:
    print("Dataset not found automatically. Please upload it:")
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            DATASET_PATH = Path(list(uploaded.keys())[0])
    except Exception:
        raise FileNotFoundError("Please upload sampled_driver_data_balanced_group_split.csv")

print(f"Loading: {DATASET_PATH}")
df = pd.read_csv(DATASET_PATH, low_memory=False)
for col in RAW_FEATURES:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"\nDataset loaded!")
print(f"  Rows     : {len(df):,}")
print(f"  Columns  : {df.shape[1]}")
print(f"  Drivers  : {df[GROUP].nunique():,}")
print(f"  Classes  : {df[TARGET].nunique()}")
df.head()

---
## Stage 4 -- Data Quality Check

Before modeling, we check for missing values. Smartphone sensors occasionally miss readings due to hardware polling gaps.

In [ ]:
# Missing Values Analysis
missing_counts = [df[c].isnull().sum() for c in RAW_FEATURES]
missing_pcts   = [df[c].isnull().mean() * 100 for c in RAW_FEATURES]

missing_df = pd.DataFrame({
    'Feature': RAW_FEATURES,
    'Missing Count': missing_counts,
    'Missing %': missing_pcts
}).sort_values('Missing %', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
colors_miss = ['#EF476F' if p > 1.0 else '#06D6A0' for p in missing_df['Missing %']]
bars = axes[0].barh(missing_df['Feature'], missing_df['Missing %'], color=colors_miss, edgecolor='white')
axes[0].set_xlabel('Missing Values (%)')
axes[0].set_title('Missing Values per Feature', fontweight='bold')
axes[0].axvline(1.0, color='red', linestyle='--', alpha=0.5, label='1% threshold')
axes[0].legend()
for bar, val in zip(bars, missing_df['Missing %']):
    axes[0].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}%', va='center', fontsize=8)

# Heatmap pattern
sample_miss = df[RAW_FEATURES].isnull().astype(int).sample(min(500, len(df)), random_state=42)
sns.heatmap(sample_miss.T, cmap=['#e8f5e9','#c62828'], cbar=False,
            ax=axes[1], yticklabels=True, xticklabels=False)
axes[1].set_title('Missing Value Pattern (500-row sample)\n(Red = missing)', fontweight='bold')
axes[1].set_xlabel('Rows (sample)')

plt.tight_layout()
plt.savefig('reports/missing_values.png', bbox_inches='tight', dpi=150)
plt.show()

print("\nMissing Value Summary:")
display(missing_df.round(4))
print("\nMissing rates < 0.3% -- sensor dropout is normal, handled by median imputation in pipeline.")

---
## Stage 5 -- Exploratory Data Analysis (EDA)

Two key questions:
1. **How severe is class imbalance?** (affects metric choice)
2. **Do sensor readings differ across driving states?** (feature utility)

In [ ]:
# EDA -- Class Distribution
class_counts = df[TARGET].value_counts()
class_pcts   = class_counts / class_counts.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

colors_cls = PALETTE[:len(class_counts)]
bars = axes[0].barh(class_counts.index, class_counts.values,
                    color=colors_cls, edgecolor='white', linewidth=0.8)
axes[0].set_xlabel('Number of Rows')
axes[0].set_title(f'Class Distribution  (Total: {len(df):,} rows)', fontweight='bold')
axes[0].invert_yaxis()
for bar, cnt, pct in zip(bars, class_counts.values, class_pcts.values):
    axes[0].text(bar.get_width() + max(class_counts.values)*0.01,
                 bar.get_y() + bar.get_height()/2,
                 f'{cnt:,}  ({pct:.1f}%)', va='center', fontsize=9)
axes[0].set_xlim(0, max(class_counts.values) * 1.25)

wedges, texts, autotexts = axes[1].pie(
    class_counts.values, labels=class_counts.index,
    colors=colors_cls, autopct='%1.1f%%',
    startangle=140, pctdistance=0.82,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for at in autotexts:
    at.set_fontsize(8)
axes[1].set_title('Class Share (%)', fontweight='bold')

plt.suptitle('Dataset Class Imbalance Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('reports/class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

print("\nImbalance Ratio (max/min class):",
      round(class_counts.max() / class_counts.min(), 1), "x")

In [ ]:
# EDA -- Sensor Patterns Across Classes
PLOT_COLS = ['speed', 'gforce', 'acc_X', 'acc_Y', 'gyroscope_data_Z']
sample = df.sample(min(20000, len(df)), random_state=42)

fig, axes = plt.subplots(1, len(PLOT_COLS), figsize=(22, 5))
for ax, col in zip(axes, PLOT_COLS):
    order = sample.groupby(TARGET)[col].median().sort_values().index
    sns.boxplot(
        data=sample, x=TARGET, y=col, order=order,
        palette=PALETTE, ax=ax, showfliers=False,
        linewidth=0.8, width=0.6
    )
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45, labelsize=7)

plt.suptitle('Sensor Reading Distributions by Driving State', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('reports/sensor_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

print("\nObservation: Speed and gforce show visible differences between Normal and dangerous states,")
print("but distributions overlap significantly -- simple thresholds won't work. ML is necessary.")

In [ ]:
# EDA -- Drivers per Class
drivers_per_class = df.groupby(TARGET)[GROUP].nunique().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(drivers_per_class.index, drivers_per_class.values,
             color=PALETTE[:len(drivers_per_class)], edgecolor='white')
axes[0].set_xlabel('Number of Unique Drivers')
axes[0].set_title('Unique Drivers Contributing to Each Class', fontweight='bold')
for i, v in enumerate(drivers_per_class.values):
    axes[0].text(v + 1, i, str(v), va='center', fontsize=9)

rows_per_driver = df.groupby(GROUP).size()
axes[1].hist(rows_per_driver, bins=40, color='#4361EE', edgecolor='white', alpha=0.85)
axes[1].axvline(rows_per_driver.median(), color='red', linestyle='--',
                label=f'Median: {rows_per_driver.median():.0f} rows')
axes[1].set_xlabel('Rows per Driver')
axes[1].set_ylabel('Number of Drivers')
axes[1].set_title('Recording Length Distribution per Driver', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('reports/driver_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Stage 6 -- Feature Engineering

**Problem:** Raw X/Y/Z accelerometer values depend on phone orientation in the car. A phone in a cupholder vs dashboard mount gives totally different axis readings for the same driving motion.

**Solution:** Orientation-invariant features:
1. **acc_total_mag** = sqrt(ax^2 + ay^2 + az^2) -- magnitude is rotation-invariant
2. **gyro_total_mag** = sqrt(gx^2 + gy^2 + gz^2) -- same idea for angular velocity  
3. **kinetic_intensity** = speed x gforce -- highlights aggressive high-speed maneuvers

In [ ]:
# Feature Engineering
df['acc_total_mag']     = np.sqrt(df['acc_X']**2 + df['acc_Y']**2 + df['acc_Z']**2)
df['gyro_total_mag']    = np.sqrt(df['gyroscope_data_X']**2 + df['gyroscope_data_Y']**2 + df['gyroscope_data_Z']**2)
df['kinetic_intensity'] = df['speed'] * df['gforce']

ENGINEERED_FEATURES = RAW_FEATURES + ['acc_total_mag', 'gyro_total_mag', 'kinetic_intensity']
MOTION_COLS  = [c for c in ENGINEERED_FEATURES if c not in ('perclos', 'mouth_open_ratio')]
STATE_COLS   = ['perclos', 'mouth_open_ratio']
COMBINED_COLS = ENGINEERED_FEATURES

print(f"Engineered 3 new features. Total: {len(ENGINEERED_FEATURES)}")

# Correlation Heatmap
corr_cols = ['speed', 'gforce', 'acc_total_mag', 'gyro_total_mag',
             'kinetic_intensity', 'perclos', 'mouth_open_ratio']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1, ax=ax,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 9}
)
ax.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('reports/correlation_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

print("\nkinetic_intensity is highly correlated with speed (expected).")
print("perclos and mouth_open_ratio are largely independent of motion features -- complementary info.")

---
## Stage 7 -- Group-Safe Train/Test Split (No Driver Leakage)

**Critical:** A naive train_test_split will put rows from the **same driver** in both sets. That driver's unique vibration signature, phone mount angle, and habits get memorized -- giving inflated test scores.

We use **GroupShuffleSplit on userid** -- every test driver was **never seen** during training.

In [ ]:
# Driver-Group Split
if 'split' in df.columns:
    train_df = df[df['split'] == 'train'].copy()
    test_df  = df[df['split'] == 'test'].copy()
    print("Using pre-computed split column.")
else:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=df[GROUP]))
    train_df = df.iloc[train_idx].copy()
    test_df  = df.iloc[test_idx].copy()

train_drivers = set(train_df[GROUP].unique())
test_drivers  = set(test_df[GROUP].unique())
overlap       = train_drivers & test_drivers

assert len(overlap) == 0, f"DATA LEAKAGE! {len(overlap)} drivers in both sets!"

X_train, y_train = train_df[COMBINED_COLS], train_df[TARGET].astype(str)
X_test,  y_test  = test_df[COMBINED_COLS],  test_df[TARGET].astype(str)

# Visualize Split
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Class distribution comparison
train_cls = train_df[TARGET].value_counts(normalize=True) * 100
test_cls  = test_df[TARGET].value_counts(normalize=True)  * 100
all_cls   = sorted(set(train_cls.index) | set(test_cls.index))
tr_vals   = [train_cls.get(c, 0) for c in all_cls]
te_vals   = [test_cls.get(c, 0)  for c in all_cls]

xp = np.arange(len(all_cls))
axes[0].bar(xp - 0.2, tr_vals, width=0.35, color='#4361EE', label='Train %', alpha=0.85)
axes[0].bar(xp + 0.2, te_vals, width=0.35, color='#F72585', label='Test %',  alpha=0.85)
axes[0].set_xticks(xp)
axes[0].set_xticklabels(all_cls, rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('Class Share (%)')
axes[0].set_title('Class Distribution: Train vs Test (Driver-Separated)', fontweight='bold')
axes[0].legend()

# Summary bars
categories = ['Train\nRows', 'Test\nRows', 'Train\nDrivers', 'Test\nDrivers', 'Overlap\nDrivers']
values     = [len(train_df), len(test_df), len(train_drivers), len(test_drivers), len(overlap)]
bar_colors = ['#4361EE', '#F72585', '#7209B7', '#3A0CA3', '#06D6A0']
bars2 = axes[1].bar(categories, values, color=bar_colors, edgecolor='white', alpha=0.85)
for bar, v in zip(bars2, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
                 f'{v:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_title('Train/Test Split Summary\n(0 driver overlap = no leakage)', fontweight='bold')
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig('reports/train_test_split.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"\nNo driver leakage confirmed.")
print(f"  Train: {len(train_df):,} rows | {len(train_drivers)} drivers")
print(f"  Test : {len(test_df):,} rows  | {len(test_drivers)} drivers")

---
## Stage 8 -- Multi-Model Benchmark

Models compared:
1. **Majority Baseline** -- always predicts most common class (floor)
2. **Logistic Regression** -- simple linear model
3. **Random Forest** -- ensemble of decision trees with class balancing
4. **ExtraTrees** -- more randomized variant of RF
5. **HistGradientBoosting** -- gradient-boosted trees (native NaN support)

In [ ]:
# Train and Evaluate All Models
models_to_eval = {
    'Majority Baseline': DummyClassifier(strategy='most_frequent'),

    'Logistic Regression': Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scl', StandardScaler()),
        ('clf', LogisticRegression(max_iter=400, class_weight='balanced', random_state=42))
    ]),

    'Random Forest': Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('clf', RandomForestClassifier(
            n_estimators=100, max_depth=16, min_samples_leaf=4,
            class_weight='balanced_subsample', n_jobs=-1, random_state=42
        ))
    ]),

    'ExtraTrees': Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('clf', ExtraTreesClassifier(
            n_estimators=100, max_depth=16, min_samples_leaf=4,
            class_weight='balanced', n_jobs=-1, random_state=42
        ))
    ]),

    'HistGradientBoosting': Pipeline([
        ('clf', HistGradientBoostingClassifier(
            max_iter=150, max_depth=10, random_state=42
        ))
    ])
}

fitted_models  = {}
benchmark_rows = []

print("Training models (this may take a few minutes)...\n")
for name, model in models_to_eval.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    elapsed = time.time() - t0
    fitted_models[name] = model

    row = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Balanced Acc': balanced_accuracy_score(y_test, preds),
        'Macro-F1': f1_score(y_test, preds, average='macro', zero_division=0),
        'Weighted-F1': f1_score(y_test, preds, average='weighted', zero_division=0),
        'Train Time (s)': round(elapsed, 2)
    }
    benchmark_rows.append(row)
    print(f"  {name:<25} | Macro-F1: {row['Macro-F1']:.4f} | Balanced Acc: {row['Balanced Acc']:.4f} | {elapsed:.1f}s")

benchmark_df = pd.DataFrame(benchmark_rows).sort_values('Macro-F1', ascending=False).reset_index(drop=True)

print("\nFull Benchmark Table:")
display(benchmark_df.round(4))

In [ ]:
# Benchmark Visualization
metrics = ['Accuracy', 'Balanced Acc', 'Macro-F1', 'Weighted-F1']
n_models = len(benchmark_df)
x = np.arange(n_models)
width = 0.19

fig, ax = plt.subplots(figsize=(15, 6))
metric_colors = ['#4361EE', '#06D6A0', '#F72585', '#FFB703']

for i, (met, col) in enumerate(zip(metrics, metric_colors)):
    bars = ax.bar(x + i * width, benchmark_df[met], width=width,
                  label=met, color=col, alpha=0.87, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=6.5, rotation=90)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(benchmark_df['Model'], fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.set_title('Multi-Model Performance Benchmark on Unseen Drivers', fontweight='bold', fontsize=13)
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('reports/model_benchmark.png', bbox_inches='tight', dpi=150)
plt.show()

best_model_name = benchmark_df.iloc[0]['Model']
best_macro_f1   = benchmark_df.iloc[0]['Macro-F1']
print(f"\nBest model: {best_model_name}  (Macro-F1 = {best_macro_f1:.4f})")

---
## Stage 9 -- Confusion Matrix Analysis

The confusion matrix shows **which classes** the best model confuses. This is more informative than any single metric.

In [ ]:
# Confusion Matrix
best_model = fitted_models[best_model_name]
preds_best = best_model.predict(X_test)
classes    = sorted(y_test.unique())

cm_raw  = confusion_matrix(y_test, preds_best, labels=classes)
cm_norm = confusion_matrix(y_test, preds_best, labels=classes, normalize='true')

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Normalized (recall per class on diagonal)
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=classes, yticklabels=classes,
    ax=axes[0], linewidths=0.4, linecolor='white',
    annot_kws={'size': 8}
)
axes[0].set_title(f'Confusion Matrix - Normalized (Recall per class)\n{best_model_name}',
                  fontweight='bold')
axes[0].set_xlabel('Predicted Class')
axes[0].set_ylabel('Actual Class')
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
axes[0].tick_params(axis='y', rotation=0,  labelsize=8)

# Raw counts
sns.heatmap(
    cm_raw, annot=True, fmt='d', cmap='Greens',
    xticklabels=classes, yticklabels=classes,
    ax=axes[1], linewidths=0.4, linecolor='white',
    annot_kws={'size': 8}
)
axes[1].set_title(f'Confusion Matrix - Raw Counts\n{best_model_name}', fontweight='bold')
axes[1].set_xlabel('Predicted Class')
axes[1].set_ylabel('Actual Class')
axes[1].tick_params(axis='x', rotation=45, labelsize=8)
axes[1].tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()
plt.savefig('reports/confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Stage 10 -- Per-Class Precision, Recall & F1

The global Macro-F1 hides important per-class variation. Here we visualize **precision, recall, and F1 for each of the 9 classes** to understand where the model struggles.

In [ ]:
# Per-Class Metrics
print(f"Classification Report - {best_model_name} (Unseen Drivers):")
print(classification_report(y_test, preds_best, digits=4, zero_division=0))

prec, rec, f1, support = precision_recall_fscore_support(
    y_test, preds_best, labels=classes, zero_division=0
)

per_class_df = pd.DataFrame({
    'Class':     classes,
    'Precision': prec,
    'Recall':    rec,
    'F1-Score':  f1,
    'Support':   support
}).sort_values('F1-Score', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

x = np.arange(len(per_class_df))
w = 0.25
axes[0].barh(x - w, per_class_df['Precision'], height=w, label='Precision', color='#4361EE', alpha=0.85)
axes[0].barh(x,     per_class_df['Recall'],    height=w, label='Recall',    color='#06D6A0', alpha=0.85)
axes[0].barh(x + w, per_class_df['F1-Score'],  height=w, label='F1-Score',  color='#F72585', alpha=0.85)
axes[0].set_yticks(x)
axes[0].set_yticklabels(per_class_df['Class'])
axes[0].set_xlabel('Score')
axes[0].set_xlim(0, 1.15)
axes[0].set_title('Per-Class Precision / Recall / F1', fontweight='bold')
axes[0].legend()
axes[0].axvline(0.5, color='gray', linestyle='--', alpha=0.5)
for i, (p, r, fv) in enumerate(zip(per_class_df['Precision'], per_class_df['Recall'], per_class_df['F1-Score'])):
    axes[0].text(p + 0.01, i - w, f'{p:.2f}', va='center', fontsize=7)
    axes[0].text(r + 0.01, i,     f'{r:.2f}', va='center', fontsize=7)
    axes[0].text(fv + 0.01, i + w, f'{fv:.2f}', va='center', fontsize=7, color='#c0145c')

# Support
support_sorted = per_class_df.sort_values('Support', ascending=True)
axes[1].barh(support_sorted['Class'], support_sorted['Support'],
             color=PALETTE[:len(support_sorted)], edgecolor='white')
axes[1].set_xlabel('Test Set Rows (Support)')
axes[1].set_title('Test Set Support per Class', fontweight='bold')
for i, v in enumerate(support_sorted['Support']):
    axes[1].text(v + 100, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('reports/per_class_metrics.png', bbox_inches='tight', dpi=150)
plt.show()

low_f1  = per_class_df[per_class_df['F1-Score'] < 0.15]['Class'].tolist()
high_f1 = per_class_df[per_class_df['F1-Score'] > 0.40]['Class'].tolist()
print(f"\nDifficult classes (F1 < 0.15): {low_f1}")
print(f"Easier classes   (F1 > 0.40): {high_f1}")

---
## Stage 11 -- Feature Importance

Which sensors contribute the most to correct predictions?

In [ ]:
# Feature Importance
rf_model  = fitted_models.get('Random Forest') or fitted_models.get('ExtraTrees')
clf_step  = rf_model.named_steps.get('clf', None)
feat_imp_df = None

if hasattr(clf_step, 'feature_importances_'):
    importances = clf_step.feature_importances_
    feat_imp_df = pd.DataFrame({
        'Feature': COMBINED_COLS,
        'Importance': importances
    }).sort_values('Importance', ascending=True)

    colors_fi = []
    for f in feat_imp_df['Feature']:
        if f in STATE_COLS:
            colors_fi.append('#F72585')
        elif f in ['acc_total_mag', 'gyro_total_mag', 'kinetic_intensity']:
            colors_fi.append('#FFB703')
        else:
            colors_fi.append('#4361EE')

    fig, ax = plt.subplots(figsize=(11, 8))
    bars = ax.barh(feat_imp_df['Feature'], feat_imp_df['Importance'],
                   color=colors_fi, edgecolor='white')
    for bar, val in zip(bars, feat_imp_df['Importance']):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8)

    legend_patches = [
        mpatches.Patch(color='#4361EE', label='Raw sensor'),
        mpatches.Patch(color='#FFB703', label='Engineered (magnitude)'),
        mpatches.Patch(color='#F72585', label='Facial/state proxy'),
    ]
    ax.legend(handles=legend_patches, loc='lower right')
    ax.set_xlabel('Feature Importance (MDI)')
    ax.set_title(f'Feature Importance - {best_model_name}', fontweight='bold', fontsize=13)
    plt.tight_layout()
    plt.savefig('reports/feature_importance.png', bbox_inches='tight', dpi=150)
    plt.show()

    top5 = feat_imp_df.tail(5)
    print("\nTop 5 Most Important Features:")
    for _, row in top5.iloc[::-1].iterrows():
        print(f"  {row['Feature']:<30} {row['Importance']:.4f} ({row['Importance']*100:.2f}%)")
else:
    print("Feature importance not available for this model.")

---
## Stage 12 -- Feature Ablation Study

**Question:** Does adding facial proxies (perclos, mouth_open_ratio) actually help on top of motion sensors?

Three configurations:
1. **Motion Only** -- speed, gforce, accelerometer, gyroscope, engineered magnitudes
2. **State Only** -- just perclos and mouth_open_ratio
3. **Combined** -- all features together

In [ ]:
# Feature Ablation Study
ablation_configs = {
    'Motion Only':  [c for c in MOTION_COLS  if c in df.columns],
    'State Only':   [c for c in STATE_COLS   if c in df.columns],
    'Combined':     [c for c in COMBINED_COLS if c in df.columns],
}

ablation_records = []
print("Running ablation study...\n")
for group_name, cols in ablation_configs.items():
    t0 = time.time()
    pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('rf',  RandomForestClassifier(
            n_estimators=80, max_depth=16, min_samples_leaf=4,
            class_weight='balanced_subsample', n_jobs=-1, random_state=42
        ))
    ])
    pipe.fit(train_df[cols], y_train)
    preds_ab = pipe.predict(test_df[cols])
    elapsed = time.time() - t0

    rec = {
        'Feature Set':    group_name,
        'Feature Count':  len(cols),
        'Accuracy':       accuracy_score(y_test, preds_ab),
        'Balanced Acc':   balanced_accuracy_score(y_test, preds_ab),
        'Macro-F1':       f1_score(y_test, preds_ab, average='macro', zero_division=0),
        'Weighted-F1':    f1_score(y_test, preds_ab, average='weighted', zero_division=0),
        'Train Time (s)': round(elapsed, 2)
    }
    ablation_records.append(rec)
    print(f"  {group_name:<15} | {len(cols)} feats | Macro-F1: {rec['Macro-F1']:.4f} | Bal.Acc: {rec['Balanced Acc']:.4f}")

ablation_df = pd.DataFrame(ablation_records).sort_values('Macro-F1', ascending=False).reset_index(drop=True)
print("\nAblation Table:")
display(ablation_df.round(4))

In [ ]:
# Ablation Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ablation_metrics = ['Accuracy', 'Balanced Acc', 'Macro-F1', 'Weighted-F1']
x_ab = np.arange(len(ablation_df))
w_ab = 0.18
ab_colors = ['#4361EE', '#06D6A0', '#F72585', '#FFB703']
for i, (met, col) in enumerate(zip(ablation_metrics, ab_colors)):
    vals = [ablation_df.iloc[j][met] for j in range(len(ablation_df))]
    bars = axes[0].bar(x_ab + i * w_ab, vals, width=w_ab, label=met, color=col, alpha=0.85, edgecolor='white')
    for bar in bars:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                     f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7.5, rotation=90)

axes[0].set_xticks(x_ab + w_ab * 1.5)
axes[0].set_xticklabels(ablation_df['Feature Set'])
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('Score')
axes[0].set_title('Ablation Study - Feature Group Comparison', fontweight='bold')
axes[0].legend()

# Macro-F1 bar
f1_vals = ablation_df.set_index('Feature Set')['Macro-F1']
groups  = ablation_df['Feature Set'].tolist()
colors_ab2 = ['#F72585', '#06D6A0', '#4361EE']
bars_ab = axes[1].bar(groups, [f1_vals.get(g, 0) for g in groups],
                      color=colors_ab2[:len(groups)], edgecolor='white', width=0.45, alpha=0.87)
for bar, g in zip(bars_ab, groups):
    v = f1_vals.get(g, 0)
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.002,
                 f'{v:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

axes[1].set_ylim(0, max(f1_vals.values) * 1.4)
axes[1].set_ylabel('Macro-F1')
axes[1].set_title('Macro-F1: Which Feature Set Wins?', fontweight='bold')

plt.tight_layout()
plt.savefig('reports/ablation_study.png', bbox_inches='tight', dpi=150)
plt.show()

print("\nConclusion: Combined features (motion + state) achieve the best Macro-F1.")
print("Facial proxies add complementary information -- multimodal approach wins.")

---
## Stage 13 -- Trip Risk Scoring (0-100)

All detected events are aggregated into a **single trip safety score** for easy interpretation:

| Event | Penalty |
|---|---|
| Drowsiness episode (EAR < 0.20) | -10 pts |
| Yawning event (MAR > 0.55) | -4 pts |
| Harsh maneuver (speed > 40 & gforce > 0.4) | -8 pts |
| Phone detected | -15 pts |

| Score Range | Rating |
|---|---|
| 85-100 | Safe |
| 70-84 | Moderate Caution |
| 0-69 | High Risk |

In [ ]:
# Trip Safety Score
def compute_safety_score(drowsy_cnt, yawn_cnt, harsh_cnt, phone_cnt=0):
    penalties = {
        'Drowsiness (eyes closed)': drowsy_cnt * 10.0,
        'Yawning':                  yawn_cnt   *  4.0,
        'Harsh maneuvers':          harsh_cnt  *  8.0,
        'Phone distraction':        phone_cnt  * 15.0
    }
    total_deductions = sum(penalties.values())
    final_score      = max(0.0, min(100.0, 100.0 - total_deductions))
    if final_score >= 85:
        rating = ('Safe', '#06D6A0')
    elif final_score >= 70:
        rating = ('Moderate Caution', '#FFB703')
    else:
        rating = ('High Risk', '#EF476F')
    return dict(score=round(final_score, 1), rating=rating, penalties=penalties,
                total_deductions=round(total_deductions, 1),
                drowsy_cnt=drowsy_cnt, yawn_cnt=yawn_cnt,
                harsh_cnt=harsh_cnt, phone_cnt=phone_cnt)

# Sensor-based event detection from test predictions
sensor_preds_series = pd.Series(preds_best, index=test_df.index)
drowsy_pred   = int((sensor_preds_series == 'Drowsiness').sum())
harsh_events  = int(((test_df['speed'] > 40) & (test_df['gforce'] > 0.4)).sum())
phone_pred    = int((sensor_preds_series == 'Using phone').sum())

# Scale to single-trip
scale = 1 / max(1, len(test_df) / 3000)
trip_report = compute_safety_score(
    drowsy_cnt = min(int(drowsy_pred * scale), 5),
    yawn_cnt   = 0,
    harsh_cnt  = min(int(harsh_events * scale), 4),
    phone_cnt  = min(int(phone_pred * scale), 3)
)

print("=" * 55)
print("       TRIP RISK ASSESSMENT")
print("=" * 55)
print(f"  Safety Score : {trip_report['score']} / 100")
print(f"  Rating       : {trip_report['rating'][0]}")
print(f"  Deductions   : -{trip_report['total_deductions']} pts")
print("\n  Penalty breakdown:")
for k, v in trip_report['penalties'].items():
    print(f"    - {k:<30} : -{v:.0f} pts")
print("=" * 55)

In [ ]:
# Trip Risk Dashboard
fig = plt.figure(figsize=(18, 8))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Gauge chart
ax_gauge = fig.add_subplot(gs[:, 0])
score = trip_report['score']
rating_label, rating_color = trip_report['rating']

theta_bg = np.linspace(np.pi, 0, 200)
ax_gauge.plot(np.cos(theta_bg), np.sin(theta_bg), color='#e0e0e0', linewidth=28, solid_capstyle='butt')

theta_score = np.linspace(np.pi, np.pi - np.pi * score / 100, 200)
ax_gauge.plot(np.cos(theta_score), np.sin(theta_score), color=rating_color,
              linewidth=28, solid_capstyle='butt')

ax_gauge.text(-1.1, -0.15, '0\nHigh Risk',  ha='center', fontsize=7.5, color='#EF476F')
ax_gauge.text( 0.0,  1.15, '50',            ha='center', fontsize=7.5, color='gray')
ax_gauge.text( 1.1, -0.15, '100\nSafe',     ha='center', fontsize=7.5, color='#06D6A0')

ax_gauge.text(0,  0.10, f'{score:.0f}',    ha='center', va='center',
              fontsize=44, fontweight='bold', color=rating_color)
ax_gauge.text(0, -0.22, rating_label,      ha='center', va='center',
              fontsize=13, fontweight='bold', color=rating_color)
ax_gauge.text(0, -0.44, 'Trip Safety Score', ha='center', va='center',
              fontsize=10, color='gray')

ax_gauge.set_xlim(-1.3, 1.3)
ax_gauge.set_ylim(-0.6, 1.4)
ax_gauge.axis('off')
ax_gauge.set_title('Trip Risk Assessment', fontweight='bold', fontsize=13)

# Penalty donut
ax_donut = fig.add_subplot(gs[0, 1])
penalties = trip_report['penalties']
non_zero = {k: v for k, v in penalties.items() if v > 0}
if non_zero:
    wedges, texts, auto = ax_donut.pie(
        non_zero.values(), labels=None,
        colors=['#EF476F','#FFB703','#4CC9F0','#F72585'],
        autopct='%1.0f%%', startangle=90,
        pctdistance=0.75,
        wedgeprops=dict(width=0.5, edgecolor='white')
    )
    ax_donut.legend(wedges, non_zero.keys(), loc='lower center',
                    bbox_to_anchor=(0.5, -0.28), fontsize=7.5)
else:
    ax_donut.text(0, 0, 'No Penalties!', ha='center', va='center',
                  fontsize=12, color='#06D6A0', fontweight='bold')
    ax_donut.axis('off')
ax_donut.set_title('Penalty Breakdown', fontweight='bold')

# Events bar
ax_events = fig.add_subplot(gs[1, 1])
event_labels = ['Drowsy\nEpisodes', 'Yawns', 'Harsh\nManeuvers', 'Phone\nUse']
event_vals   = [trip_report['drowsy_cnt'], trip_report['yawn_cnt'],
                trip_report['harsh_cnt'],  trip_report['phone_cnt']]
event_colors = ['#EF476F', '#FFB703', '#4CC9F0', '#F72585']
bars_ev = ax_events.bar(event_labels, event_vals, color=event_colors, edgecolor='white')
for bar, v in zip(bars_ev, event_vals):
    ax_events.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
                   str(v), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax_events.set_ylabel('Count')
ax_events.set_title('Detected Events', fontweight='bold')

# Prediction distribution
ax_cls = fig.add_subplot(gs[:, 2])
pred_counts = pd.Series(preds_best).value_counts()
top_n = pred_counts.head(6)
others = pred_counts.iloc[6:].sum()
if others > 0:
    top_n['Others'] = others

wedges2, texts2, auto2 = ax_cls.pie(
    top_n.values, labels=top_n.index,
    colors=PALETTE[:len(top_n)],
    autopct='%1.1f%%', startangle=140,
    pctdistance=0.8,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for t in auto2:
    t.set_fontsize(8)
ax_cls.set_title(f'Sensor Prediction Distribution\n({len(preds_best):,} test rows)',
                 fontweight='bold')

plt.suptitle('Driver Safety Dashboard', fontsize=15, fontweight='bold', y=1.01)
plt.savefig('reports/trip_risk_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Stage 14 -- Final Results Summary Dashboard

In [ ]:
# Final Results Summary Dashboard
fig = plt.figure(figsize=(20, 14))
gs2 = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.35)

# Model benchmark
ax1 = fig.add_subplot(gs2[0, :2])
bdf_sorted = benchmark_df.sort_values('Macro-F1')
colors_bench = ['#EF476F' if m == 'Majority Baseline' else '#4361EE' for m in bdf_sorted['Model']]
bars1 = ax1.barh(bdf_sorted['Model'], bdf_sorted['Macro-F1'],
                 color=colors_bench, edgecolor='white', height=0.55)
for bar, idx in zip(bars1, bdf_sorted.index):
    row = bdf_sorted.loc[idx]
    ax1.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
             f'Macro-F1: {row["Macro-F1"]:.4f}  |  Bal.Acc: {row["Balanced Acc"]:.4f}',
             va='center', fontsize=9)
ax1.set_xlabel('Macro-F1')
ax1.set_title('Model Comparison -- Macro-F1 on Unseen Drivers', fontweight='bold')

# Summary card
ax2 = fig.add_subplot(gs2[0, 2])
ax2.axis('off')
best_row     = benchmark_df.iloc[0]
baseline_row = benchmark_df[benchmark_df['Model']=='Majority Baseline'].iloc[0]
summary_text = (
    f"PROJECT RESULTS SUMMARY\n"
    f"{'-'*32}\n"
    f"Dataset    : {len(df):,} rows\n"
    f"Drivers    : {df[GROUP].nunique():,}\n"
    f"Classes    : 9\n"
    f"Train rows : {len(train_df):,}\n"
    f"Test rows  : {len(test_df):,}\n"
    f"Leakage    : 0 drivers\n\n"
    f"Best Model : {best_row['Model']}\n"
    f"Macro-F1   : {best_row['Macro-F1']:.4f}\n"
    f"Bal.Acc    : {best_row['Balanced Acc']:.4f}\n"
    f"Accuracy   : {best_row['Accuracy']:.4f}\n\n"
    f"vs Baseline:\n"
    f"  Macro-F1 : {baseline_row['Macro-F1']:.4f}\n"
    f"  Bal.Acc  : {baseline_row['Balanced Acc']:.4f}\n\n"
    f"Improvement:\n"
    f"  dMacro-F1: +{best_row['Macro-F1']-baseline_row['Macro-F1']:.4f}\n"
    f"  dBal.Acc : +{best_row['Balanced Acc']-baseline_row['Balanced Acc']:.4f}"
)
ax2.text(0.05, 0.97, summary_text, transform=ax2.transAxes,
         fontsize=9.5, va='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#f0f4ff', edgecolor='#4361EE', alpha=0.9))

# Ablation
ax3 = fig.add_subplot(gs2[1, :2])
ab_sorted = ablation_df.sort_values('Macro-F1')
x3 = np.arange(len(ab_sorted))
w3 = 0.25
ax3.bar(x3 - w3, ab_sorted['Macro-F1'],    width=w3, label='Macro-F1',    color='#F72585', alpha=0.85)
ax3.bar(x3,      ab_sorted['Balanced Acc'], width=w3, label='Balanced Acc', color='#06D6A0', alpha=0.85)
ax3.bar(x3 + w3, ab_sorted['Accuracy'],    width=w3, label='Accuracy',    color='#4361EE', alpha=0.85)
ax3.set_xticks(x3)
ax3.set_xticklabels(ab_sorted['Feature Set'])
ax3.set_ylim(0, 0.9)
ax3.set_ylabel('Score')
ax3.set_title('Ablation Study -- Macro-F1 / Balanced Acc / Accuracy per Feature Group', fontweight='bold')
ax3.legend()

# Top features
ax4 = fig.add_subplot(gs2[1, 2])
if feat_imp_df is not None:
    top_feats = feat_imp_df.tail(7)
    f_colors = ['#F72585' if f in STATE_COLS else ('#FFB703' if f in ['acc_total_mag','gyro_total_mag','kinetic_intensity'] else '#4361EE')
                for f in top_feats['Feature']]
    ax4.barh(top_feats['Feature'], top_feats['Importance'], color=f_colors, edgecolor='white')
    ax4.set_title('Top 7 Features (MDI Importance)', fontweight='bold')
    ax4.set_xlabel('Importance')
else:
    ax4.axis('off')
    ax4.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax4.transAxes)

# Per-class F1 heatmap across all models
ax5 = fig.add_subplot(gs2[2, :])
f1_matrix = {}
for name, mdl in fitted_models.items():
    if name == 'Majority Baseline':
        continue
    p_ = mdl.predict(X_test)
    _, _, f1_, _ = precision_recall_fscore_support(y_test, p_, labels=classes, zero_division=0)
    f1_matrix[name] = f1_

f1_heatmap_df = pd.DataFrame(f1_matrix, index=classes).T
sns.heatmap(
    f1_heatmap_df, annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=0, vmax=1, ax=ax5,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 9}
)
ax5.set_title('Per-Class F1-Score Across All Models  (Green = Good / Red = Poor)', fontweight='bold')
ax5.set_xlabel('Driving State Class')
ax5.set_ylabel('Model')
ax5.tick_params(axis='x', rotation=30, labelsize=9)
ax5.tick_params(axis='y', rotation=0)

plt.suptitle('Driver Behavior Analysis -- Complete Results Summary', fontsize=16, fontweight='bold', y=1.01)
plt.savefig('reports/final_summary_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()

print("\nAll figures saved to /reports/")

---
## Conclusions & Limitations

### Key Findings

| Finding | Evidence |
|---|---|
| Accuracy is misleading on imbalanced data | Dummy model: ~27% acc but 0.047 Macro-F1 |
| Driver leakage inflates scores | GroupShuffleSplit on userid ensures true unseen-driver evaluation |
| Inertial sensors have limits | Smoking/phone detection near-zero recall; cabin events need video |
| Combined modalities work best | Motion + facial features beat either alone in ablation study |
| Class imbalance is severe | Imbalance ratio up to 50x -- class-weighted models essential |
| Random Forest best overall | Highest Macro-F1 and balanced accuracy in benchmark |

### Limitations

1. Row-level classification -- temporal context (sliding windows, LSTM) could be better
2. Rare classes (Smoking, Eating) have too few drivers for robust generalization
3. Camera-off bias -- class label concentration at driver level causes test set distribution shift
4. Research prototype only -- not validated as a safety system
5. Privacy -- facial landmarks should be computed on-device; raw video should not be stored

### What Next?

- **Temporal models** -- LSTM/TCN on 5-second sliding windows
- **ONNX export** -- mobile deployment on Android
- **SMOTE** -- synthetic oversampling for rare classes
- **Multi-split cross-validation** -- more robust performance estimates
- **Real-time inference** -- integrate sensor + video pipelines

In [ ]:
# Save Best Model
import joblib
model_path = Path('models/best_model.pkl')
joblib.dump(fitted_models[best_model_name], model_path)

print("=" * 60)
print("  PROJECT COMPLETE -- FINAL SUMMARY")
print("=" * 60)
print(f"  Dataset        : {len(df):,} rows, {df[GROUP].nunique()} drivers")
print(f"  Classes        : 9 driving behavior categories")
print(f"  Best model     : {best_model_name}")
print(f"  Macro-F1       : {benchmark_df.iloc[0]['Macro-F1']:.4f}")
print(f"  Balanced Acc   : {benchmark_df.iloc[0]['Balanced Acc']:.4f}")
print(f"  Driver leakage : 0 (GroupShuffleSplit enforced)")
print(f"  Model saved    : {model_path}")
print("\n  Reports saved:")
for p in sorted(Path('reports').glob('*.png')):
    print(f"    {p}")
print("=" * 60)